# utils

> Time formatting utilities for VAD alignment display

In [ ]:
#| default_exp utils

In [ ]:
#| export
from typing import Optional, List, Set, TYPE_CHECKING

if TYPE_CHECKING:
    from cjm_transcript_vad_align.models import VADChunk

## Time Formatting

In [ ]:
#| export
def format_time_precise(
    seconds: Optional[float]  # Time in seconds
) -> str:  # Formatted time string (m:ss.s)
    """Format seconds as m:ss.s for sub-second display."""
    if seconds is None:
        return "-:--.-"
    minutes = int(seconds // 60)
    secs = seconds % 60
    return f"{minutes}:{secs:04.1f}"

## Tests

## Audio File Boundaries

In [ ]:
#| export
def get_audio_file_boundaries(
    chunks: List["VADChunk"],  # Ordered list of VAD chunks
) -> Set[int]:  # Indices where audio_file_index changes from the previous chunk
    """Find indices where audio_file_index changes between adjacent chunks.
    
    A boundary at index N means chunk[N].audio_file_index differs from
    chunk[N-1].audio_file_index.
    """
    boundaries = set()
    for i in range(1, len(chunks)):
        if chunks[i].audio_file_index != chunks[i - 1].audio_file_index:
            boundaries.add(i)
    return boundaries

In [ ]:
#| export
def get_audio_file_count(
    chunks: List["VADChunk"],  # Ordered list of VAD chunks
) -> int:  # Number of unique audio files
    """Count the number of unique audio files in the chunk list."""
    if not chunks:
        return 0
    return len({c.audio_file_index for c in chunks})

In [ ]:
#| export
def get_audio_file_position(
    chunks: List["VADChunk"],  # Ordered list of VAD chunks
    focused_index: int,  # Index of the focused chunk
) -> Optional[int]:  # 1-based position of the audio file, or None
    """Get the audio file position (1-based) of the focused chunk.
    
    Returns which audio file the focused chunk belongs to,
    based on order of first appearance.
    """
    if not chunks or focused_index < 0 or focused_index >= len(chunks):
        return None
    focused_afi = chunks[focused_index].audio_file_index
    seen = []
    for c in chunks:
        if c.audio_file_index not in seen:
            seen.append(c.audio_file_index)
    if focused_afi in seen:
        return seen.index(focused_afi) + 1
    return None

In [ ]:
assert format_time_precise(6.6) == "0:06.6"
assert format_time_precise(9.8) == "0:09.8"
assert format_time_precise(65.25) == "1:05.2"
assert format_time_precise(0.0) == "0:00.0"
assert format_time_precise(None) == "-:--.-"
print("format_time_precise tests passed")

In [ ]:
from cjm_transcript_vad_align.models import VADChunk

# Test get_audio_file_boundaries
chunks_single = [
    VADChunk(index=0, start_time=0.0, end_time=1.0, audio_file_index=0),
    VADChunk(index=1, start_time=1.0, end_time=2.0, audio_file_index=0),
]
assert get_audio_file_boundaries(chunks_single) == set()

chunks_multi = [
    VADChunk(index=0, start_time=0.0, end_time=1.0, audio_file_index=0),
    VADChunk(index=1, start_time=1.0, end_time=2.0, audio_file_index=0),
    VADChunk(index=2, start_time=0.0, end_time=1.5, audio_file_index=1),
    VADChunk(index=3, start_time=1.5, end_time=3.0, audio_file_index=1),
    VADChunk(index=4, start_time=0.0, end_time=2.0, audio_file_index=2),
]
assert get_audio_file_boundaries(chunks_multi) == {2, 4}
assert get_audio_file_boundaries([]) == set()
print("get_audio_file_boundaries tests passed")

# Test get_audio_file_count
assert get_audio_file_count(chunks_single) == 1
assert get_audio_file_count(chunks_multi) == 3
assert get_audio_file_count([]) == 0
print("get_audio_file_count tests passed")

# Test get_audio_file_position
assert get_audio_file_position(chunks_multi, 0) == 1
assert get_audio_file_position(chunks_multi, 1) == 1
assert get_audio_file_position(chunks_multi, 2) == 2
assert get_audio_file_position(chunks_multi, 4) == 3
assert get_audio_file_position(chunks_multi, 99) is None
assert get_audio_file_position([], 0) is None
print("get_audio_file_position tests passed")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()